<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/seminars/3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое задание №3

## Свёртка и корреляционный анализ сигналов

## Цели занятия
- Освоить вычисление свёртки и корреляции в Python.
- Применить свёртку для фильтрации сигналов.
- Использовать кросс-корреляцию для поиска временной задержки и обнаружения шаблона.
- Проанализировать влияние уровня шума на точность оценок.

## Подготовка окружения

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from IPython.display import Audio, display
import time

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
```

---

## Задание 1. Свёртка гауссовых функций: сравнение численного и аналитического результатов

### Теоретическое введение
Свёртка двух гауссовых функций даёт гауссову функцию с дисперсией, равной сумме дисперсий. Если
\[
f(t) = \frac{1}{\sqrt{2\pi\sigma_1^2}} e^{-t^2/(2\sigma_1^2)}, \quad g(t) = \frac{1}{\sqrt{2\pi\sigma_2^2}} e^{-t^2/(2\sigma_2^2)},
\]
то
\[
(f * g)(t) = \frac{1}{\sqrt{2\pi(\sigma_1^2+\sigma_2^2)}} e^{-t^2/(2(\sigma_1^2+\sigma_2^2))}.
\]

### Задание
1. Сгенерируйте дискретные гауссовы импульсы (например, с помощью `scipy.signal.windows.gaussian`) с заданными стандартными отклонениями `sigma1=3`, `sigma2=5`. Используйте длину окна, достаточную для захвата всей значимой части (например, 10*sigma).
2. Вычислите свёртку численно с помощью `np.convolve`.
3. Постройте графики:
   - Исходные функции.
   - Результат свёртки (численный).
   - Теоретическую гауссову функцию с дисперсией `sigma1^2 + sigma2^2`.
4. Оцените среднеквадратичную ошибку между численным и теоретическим результатами.

```python
# Ваш код здесь
```

**Вопросы:**
- Почему амплитуда свёртки отличается от амплитуд исходных функций?
- Что произойдёт с формой свёртки при увеличении σ?

---

## Задание 2. Фильтрация с помощью свёртки: сравнение прямоугольного и гауссовского окон

### Цель
Изучить, как выбор ядра и его длина влияют на подавление высокочастотной составляющей и сохранение низкочастотной.

### Задание
1. Создайте сигнал длительностью 2 секунды, частота дискретизации 1000 Гц, состоящий из суммы двух синусоид:
   - низкая частота \( f_1 = 5 \) Гц, амплитуда 1,
   - высокая частота \( f_2 = 80 \) Гц, амплитуда 0.5.
2. Сгенерируйте прямоугольное окно длины `L` (например, `L = 21`), нормализованное так, чтобы сумма коэффициентов была 1.
3. Сгенерируйте гауссовское окно той же длины (используйте `signal.windows.gaussian(L, std=5)`), также нормализованное.
4. Примените свёртку с этими окнами (используйте `mode='same'`).
5. Постройте графики:
   - Исходный сигнал (первые 0.5 с).
   - Отфильтрованные сигналы для обоих окон.
6. Вычислите и сравните амплитуды полезной составляющей (5 Гц) и подавленной (80 Гц) после фильтрации. Для этого:
   - Возьмите БПФ сигналов,
   - Измерьте амплитуды на соответствующих частотах.
7. Исследуйте влияние длины окна: повторите для длин `L = 11, 21, 41, 81`. Постройте графики зависимости амплитуды 80 Гц и 5 Гц от длины окна для обоих типов окон.

```python
# Ваш код здесь
```

**Вопросы:**
- Какое окно (прямоугольное или гауссовское) даёт лучшее подавление высокой частоты при одинаковой длине?
- Как увеличение длины окна влияет на сохранение амплитуды низкой частоты?
- Почему гауссовское окно имеет более плавную частотную характеристику?

---

## Задание 3. Поиск временной задержки с помощью кросс-корреляции

### Цель
Определить временной сдвиг между двумя сигналами в присутствии шума и оценить влияние уровня шума на точность.

### Задание
1. Сгенерируйте сигнал длительностью 1 секунду (fs=1000 Гц), представляющий собой сумму 3 синусоид со случайными частотами, амплитудами и фазами (частоты выберите в диапазоне 10–100 Гц). Назовём его `x`.
2. Создайте второй сигнал `y`, который является сдвинутой во времени копией `x` на `delay` отсчётов (выберите задержку, например, 100 отсчётов) и добавьте к нему белый гауссов шум с уровнем `noise_level` (например, 0.1).
3. Вычислите кросс-корреляцию `corr = np.correlate(x, y, mode='full')` и найдите индекс максимума. Оцените задержку как `delay_est = argmax(corr) - (len(x)-1)` (поскольку `mode='full'` даёт диапазон от -N+1 до N-1).
4. Повторите эксперимент для различных уровней шума (например, `noise_level = 0, 0.1, 0.5, 1, 2`). Для каждого уровня выполните 50 экспериментов со случайными реализациями шума и постройте график средней абсолютной ошибки `|delay_est - delay|` в зависимости от уровня шума.
5. Для одного выбранного уровня шума (например, 0.5) постройте график кросс-корреляции и отметьте положение истинной и найденной задержки.

```python
# Ваш код здесь
```

**Вопросы:**
- Как зависит точность оценки задержки от уровня шума?
- Что произойдёт, если сигнал будет иметь резкие пики (например, прямоугольные импульсы)? Будет ли точность выше?
- Почему при высоком уровне шума могут появляться ложные пики?

---

## Задание 4. Обнаружение шаблона в зашумлённом сигнале

### Цель
Найти местоположение сигнала, имеющего форму гауссова импульса, модулированного синусоидой, в смеси с шумом. Исследовать влияние отношения сигнал/шум на точность обнаружения.

### Задание
1. Создайте шаблон `template` – произведение гауссовой огибающей (σ=10 отсчётов) на синусоиду частотой 20 Гц (при fs=1000 Гц). Длина шаблона примерно 5σ.
2. Создайте длинный сигнал `long_signal` длиной 2000 отсчётов, состоящий из:
   - белого гауссова шума с единичной дисперсией,
   - вставленного в случайную позицию шаблона, умноженного на амплитуду `A` (например, 2).
3. Используйте кросс-корреляцию `np.correlate(long_signal, template, mode='valid')` для поиска позиции шаблона. Найдите индекс максимума корреляции.
4. Повторите эксперимент для разных отношений сигнал/шум (SNR), варьируя амплитуду шаблона от 0.2 до 5. Для каждого SNR вычислите среднеквадратичную ошибку позиции (абсолютное отклонение от истинной позиции) по 50 экспериментам (с разными реализациями шума и случайными позициями). Постройте график зависимости ошибки от SNR.
5. Для одного значения SNR (например, 1) визуализируйте: исходный длинный сигнал, шаблон, результат кросс-корреляции с отмеченным пиком.

```python
# Ваш код здесь
```

**Вопросы:**
- Как форма шаблона влияет на точность обнаружения?
- Почему при низком SNR ошибка резко возрастает?
- Какие методы можно использовать для повышения надёжности обнаружения?

---

## Задание 5. Поиск фрагмента в реальном аудиосигнале

### Цель
Применить кросс-корреляцию для нахождения заданного фрагмента в аудиофайле. Фрагмент и основной файл предоставлены (студентам нужно будет загрузить их).

### Задание
1. Загрузите аудиофайл `full_audio.wav` (например, запись речи или музыки) и фрагмент `fragment.wav` (короткий отрезок из этого же файла). Используйте `scipy.io.wavfile.read`.
2. Приведите сигналы к моно (если стерео) и нормализуйте их в диапазон [-1, 1].
3. Вычислите кросс-корреляцию между полным сигналом и фрагментом.
4. Найдите позицию максимального значения корреляции и определите временное смещение (в отсчётах и в секундах).
5. Постройте график кросс-корреляции и отметьте найденный пик.
6. Вырежьте из полного сигнала участок, соответствующий найденной позиции, и прослушайте его (используйте `Audio`). Убедитесь, что он совпадает с фрагментом.

```python
# Загрузка файлов (студенты могут загрузить через файловый менеджер Colab или указать пути)
# Для демонстрации можно сгенерировать тестовые данные, но лучше использовать реальные файлы.

# Если файлы находятся в рабочей директории:
# rate_full, full = wavfile.read('full_audio.wav')
# rate_frag, frag = wavfile.read('fragment.wav')

# Для Colab можно использовать загрузку:
# from google.colab import files
# uploaded = files.upload()  # выбрать full_audio.wav
# uploaded = files.upload()  # выбрать fragment.wav

# Ваш код здесь
```

**Вопросы:**
- Почему перед вычислением корреляции сигналы следует нормализовать?
- Что будет, если фрагмент не содержится в полном сигнале? Как это отразится на корреляции?
- Какие ограничения имеет метод поиска по корреляции для реальных аудиосигналов?

---

## Подведение итогов

По завершении всех заданий оформите ноутбук, добавив текстовые ответы на вопросы. Сохраните файл как `Seminar3_Фамилия.ipynb`.

```python
# Для проверки себя: запустите все ячейки и убедитесь, что графики и выводы корректны.
```

## Дополнительные замечания для преподавателя

- В заданиях предусмотрены места для кода. Студенты должны написать недостающие части.
- Для заданий 3 и 4 рекомендуется использовать `np.random.seed` для воспроизводимости результатов.
- В задании 5 можно предложить студентам записать собственный фрагмент (например, голосом) и искать его в записи.

Если нужны конкретные wav-файлы для задания 5, их можно подготовить заранее и выложить в репозиторий курса.